# Baseline RT-Splatting trên Colab: VSGD scene 0758

Notebook này chạy **RT-Splatting** (baseline của repo: `train.py` → `render.py` → `metrics.py`) trên đúng scene, split, độ phân giải, mask và cách tính metric của lượt chạy VIGOR-GS trong `colab_vsgd_0758.ipynb`, rồi đặt kết quả cạnh 6 biến thể VIGOR-GS trong cùng một bảng.

Mục đích là trả lời câu hỏi: kết quả của VIGOR-GS có thật sự đến từ cơ chế mới, hay từ những phần kế thừa (2DGS với `p·o` của RT-Splatting, deformation của Deformable-3DGS, ảnh clean của mạng xoá phản chiếu)?

**Cần chuẩn bị** giống notebook VIGOR-GS:

1. Nhánh `viet_huynh` trên GitHub (repo công khai `iamkyledang/vigor_gs`).
2. Zip của folder `0758` trên Drive (mặc định `MyDrive/0758.zip`).
3. Runtime có GPU.

`RUN_ROOT` mặc định **trùng** với notebook VIGOR-GS. Nhờ vậy notebook này dùng lại wheel đã build trong `RUN_ROOT/_wheels/` và đọc được `results.json` của các biến thể VIGOR-GS để so sánh. Kết quả baseline nằm riêng trong `RUN_ROOT/rtsplatting/`, không đụng vào thư mục nào của VIGOR-GS.

### Điều kiện so sánh

| Mục | RT-Splatting (notebook này) | VIGOR-GS `full` |
|---|---|---|
| Split | `colmap/split.json`: 117 train / 29 test | như nhau |
| Ảnh | `colmap/images`, `-r 2` (964×540) | như nhau |
| Mask kính | `colmap/masks`, dùng trong loss | như nhau |
| Ảnh clean | **không dùng** (`--ignore_clean_images`) | Phase I học theo ảnh clean |
| Không gian màu khi train | sRGB (mặc định của RT-Splatting) | linear RGB |
| Số iteration | 61 000 (mặc định của RT-Splatting) | 30 000 + 30 000 = 60 000 |
| Thời gian | tĩnh, không có deformation | deformation MLP theo `t` |
| Metric | `metrics.py` trên PNG sRGB | như nhau |

Hai điểm cần ghi rõ khi báo cáo:

- **`env_scope` của RT-Splatting.** Trong `eval.sh`, mỗi scene có tâm và bán kính `env_scope` được chỉnh tay. Mặc định ở đây là không giới hạn (`ENV_SCOPE = None`), tức mọi Gaussian đều được phép có reflectance và transmissivity. Ô "Gợi ý `env_scope`" bên dưới tính một giá trị từ dữ liệu nếu bạn muốn baseline được chỉnh kỹ hơn. Nếu chạy cả hai cấu hình, hãy đổi `BASELINE_NAME` để chúng không ghi đè lên nhau.
- **RT-Splatting là phương pháp tĩnh.** VSGD là video có người đi lại, nên baseline chịu thiệt ở các vùng chuyển động. Muốn tách ảnh hưởng của deformation thì cần thêm Deformable-3DGS làm baseline thứ hai.

In [ ]:
# ⚠️ ĐIỀN / KIỂM TRA các giá trị trong ô này
REPO = 'iamkyledang/vigor_gs'   # repo công khai trên GitHub
BRANCH = 'viet_huynh'

ZIP_PATH = '/content/drive/MyDrive/0758.zip'                    # ⚠️ KIỂM TRA: zip của folder 0758 trên Drive
RUN_ROOT = '/content/drive/MyDrive/vigor_gs_runs/VSGD_0758'     # trùng notebook VIGOR-GS để dùng lại cache và so sánh
BASELINE_NAME = 'rtsplatting'                                   # kết quả nằm ở RUN_ROOT/<BASELINE_NAME>

# Cấu hình baseline. Đổi sau khi đã chạy thì phải đổi BASELINE_NAME (notebook sẽ báo lỗi nếu lệch).
RESOLUTION = 2                     # phải giống lượt chạy VIGOR-GS
ITERS = 61_000                     # mặc định của RT-Splatting; VIGOR-GS full dùng tổng 60 000
INIT_UNTIL_ITER = 700              # giá trị dùng cho phần lớn scene thật trong eval.sh
NORM_LOSS_FROM_ITER = 700
ENV_SCOPE = None                   # None = không giới hạn; hoặc [cx, cy, cz, bán kính] (xem ô "Gợi ý env_scope")
XYZ_AXIS = [2, 1, 0]               # thứ tự trục cho environment map; RT-Splatting chỉ dùng khi có ENV_SCOPE
DATA_DEVICE = 'cuda'               # 'cpu' nếu thiếu VRAM
EXTRA = []                         # tham số thêm cho train.py

REPO_DIR = '/content/vigor_gs'
SCENE_DIR = '/content/data/VSGD/0758'

In [ ]:
import os
import subprocess

print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], capture_output=True, text=True).stdout)

from google.colab import drive

drive.mount('/content/drive')
os.makedirs(RUN_ROOT, exist_ok=True)

## Lấy code từ nhánh `viet_huynh`

In [ ]:
import os
import subprocess


def _git(*args):
    result = subprocess.run(['git', *args], capture_output=True, text=True, env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})  # không treo chờ tài khoản
    if result.returncode:
        raise RuntimeError(f'git {args[0]} thất bại:\n{result.stderr or result.stdout}')
    return result.stdout.strip()


url = f'https://github.com/{REPO}.git'  # repo công khai, không cần token
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    _git('-C', REPO_DIR, 'fetch', '-q', url, BRANCH)
    _git('-C', REPO_DIR, 'reset', '-q', '--hard', 'FETCH_HEAD')
else:
    _git('clone', '-q', '--branch', BRANCH, '--single-branch', url, REPO_DIR)
CODE_VERSION = _git('-C', REPO_DIR, 'log', '-1', '--format=%h %s (%ci)')
print('Code:', CODE_VERSION)

## Cài thư viện và build các CUDA extension

Giống hệt notebook VIGOR-GS, nên nếu đã chạy notebook đó với cùng `RUN_ROOT` thì các wheel được lấy từ cache trong vài giây. RT-Splatting cần thêm `nvdiffrast` (đã nằm trong danh sách build) và `open3d` (`render.py` import nó dù ta bỏ bước trích mesh bằng `--skip_mesh`).

In [ ]:
import glob
import importlib
import importlib.util
import os
import shutil
import subprocess
import sys

import torch


def pip(*args, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *args], check=check).returncode


pip('install', '-q', 'plyfile', 'mediapy', 'ninja', 'setuptools==81.0.0')


def ensure_include(rel_path, header):
    """Some newer gcc / CUDA versions need these includes; adding them is harmless otherwise."""
    path = os.path.join(REPO_DIR, rel_path)
    with open(path) as f:
        source = f.read()
    if f'#include <{header}>' not in source:
        with open(path, 'w') as f:
            f.write(f'#include <{header}>\n' + source)


ensure_include('submodules/simple-knn/simple_knn.cu', 'cfloat')
ensure_include('submodules/diff-surfel-anych/cuda_rasterizer/rasterizer_impl.h', 'cstdint')

major, minor = torch.cuda.get_device_capability(0)
tree = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD:submodules'], capture_output=True, text=True, check=True).stdout.strip()
tag = f"torch{torch.__version__.replace('+', '-')}_py{sys.version_info.major}{sys.version_info.minor}_sm{major}{minor}_{tree}"
WHEEL_DIR = os.path.join(RUN_ROOT, '_wheels', tag)
if not os.path.exists(os.path.join(WHEEL_DIR, '.complete')):
    build_dir = '/content/_wheels_build'
    shutil.rmtree(build_dir, ignore_errors=True)
    os.environ['TORCH_CUDA_ARCH_LIST'] = f'{major}.{minor}'
    for source in ('submodules/simple-knn', 'submodules/diff-surfel-anych', 'submodules/diff-gaussian-rasterization', 'git+https://github.com/NVlabs/nvdiffrast'):
        print('Build', source, flush=True)
        pip('wheel', '-q', '--no-build-isolation', '--no-deps', '-w', build_dir, source if source.startswith('git+') else os.path.join(REPO_DIR, source))
    os.makedirs(WHEEL_DIR, exist_ok=True)
    for wheel in glob.glob(os.path.join(build_dir, '*.whl')):
        shutil.copy(wheel, WHEEL_DIR)
    open(os.path.join(WHEEL_DIR, '.complete'), 'w').close()
pip('install', '-q', '--no-deps', '--force-reinstall', *glob.glob(os.path.join(WHEEL_DIR, '*.whl')))

for module in ('simple_knn._C', 'diff_surfel_anych', 'nvdiffrast.torch', 'cv2'):
    importlib.import_module(module)

# render.py import open3d ở đầu file. Ta không trích mesh (--skip_mesh), nên nếu không cài được open3d
# thì một module rỗng là đủ để import chạy qua.
STUB_DIR = '/content/_stubs'
if importlib.util.find_spec('open3d') is None and pip('install', '-q', 'open3d', check=False) != 0:
    os.makedirs(STUB_DIR, exist_ok=True)
    with open(os.path.join(STUB_DIR, 'open3d.py'), 'w') as f:
        f.write('# stub: open3d is only needed for mesh extraction, which this notebook skips\n')
    print('Không cài được open3d: dùng module rỗng (chỉ ảnh hưởng việc trích mesh, notebook không dùng).')
print('OK:', 'torch', torch.__version__, '| CUDA', torch.version.cuda, '|', torch.cuda.get_device_name(0))

## Dữ liệu: giải nén `colmap/` của scene 0758 và kiểm tra

Giống notebook VIGOR-GS, nhưng phần kiểm tra không đòi ảnh clean vì baseline không dùng chúng.

In [ ]:
import json
import os
import shutil
import zipfile

from PIL import Image


def extract_colmap(zip_path, scene_dir):
    """Extract only <prefix>colmap/ of the scene zip into scene_dir/colmap/."""
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f'Không thấy {zip_path}; kiểm tra ZIP_PATH ở ô cấu hình')
    stamp = f'{os.path.getsize(zip_path)}-{int(os.path.getmtime(zip_path))}'
    marker = os.path.join(scene_dir, 'colmap', '.extracted_from_zip')
    if os.path.exists(marker) and open(marker).read() == stamp:
        print('Đã giải nén trước đó, bỏ qua:', os.path.join(scene_dir, 'colmap'))
        return
    with zipfile.ZipFile(zip_path) as z:
        entries = [(info, info.filename.replace('\\', '/')) for info in z.infolist()]
        hits = [name for _, name in entries if name.endswith('colmap/frames.json')]
        if len(hits) != 1:
            raise RuntimeError(f'Cần đúng một colmap/frames.json trong {zip_path}, tìm thấy {len(hits)}: {hits[:3]}')
        prefix = hits[0][: -len('colmap/frames.json')]
        shutil.rmtree(os.path.join(scene_dir, 'colmap'), ignore_errors=True)
        count = 0
        for info, name in entries:
            if not name.startswith(prefix + 'colmap/') or name.endswith('/'):
                continue
            target = os.path.join(scene_dir, name[len(prefix):])
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(info) as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            count += 1
    with open(marker, 'w') as f:
        f.write(stamp)
    print(f'Đã giải nén {count} file từ "{prefix}colmap/" vào {os.path.join(scene_dir, "colmap")}')


def check_scene(scene_dir):
    """Every frame of split.json needs its COLMAP image and an undistorted mask of the same size."""
    colmap = os.path.join(scene_dir, 'colmap')
    with open(os.path.join(colmap, 'split.json')) as f:
        split = json.load(f)
    problems = [f'thiếu colmap/sparse/0/{name}' for name in ('cameras.bin', 'images.bin', 'points3D.bin') if not os.path.exists(os.path.join(colmap, 'sparse', '0', name))]
    sizes = set()
    for split_name in ('train', 'test'):
        for name in split[split_name]:
            paths = {'ảnh': os.path.join(colmap, 'images', name), 'mask': os.path.join(colmap, 'masks', os.path.splitext(name)[0] + '.png')}
            for kind, path in paths.items():
                if not os.path.exists(path):
                    problems.append(f'thiếu {kind}: {os.path.relpath(path, scene_dir)}')
            if name == split[split_name][0] and all(os.path.exists(p) for p in paths.values()):
                sizes |= {Image.open(p).size for p in paths.values()}
    if len(sizes) > 1:
        problems.append(f'ảnh và mask khác kích thước: {sizes}')
    if problems:
        raise RuntimeError(f'{len(problems)} vấn đề với dữ liệu:\n  ' + '\n  '.join(problems[:10]))
    print(f'Dữ liệu OK: {len(split["train"])} train / {len(split["test"])} test; ảnh và mask đều {sizes.pop()}')


extract_colmap(ZIP_PATH, SCENE_DIR)
check_scene(SCENE_DIR)

## Gợi ý `env_scope` (không bắt buộc)

RT-Splatting coi các Gaussian **ngoài** khối cầu `env_scope` là môi trường: chúng không có reflectance và transmissivity. `eval.sh` chỉnh tay tâm và bán kính cho từng scene. Ô này đưa ra một gợi ý có thể giải thích được, dựa trên lượt chạy VIGOR-GS `full`:

- **tâm** = trung vị các điểm kính trong `RUN_ROOT/full/glass_points.ply` (điểm do `extract_plane.py` gom từ mặt kính);
- **bán kính** = trung vị khoảng cách từ tâm đó tới các camera train. Khối cầu khi đó chứa tấm kính và những gì phía sau nó, còn phần cảnh phía sau camera (nơi sinh ra phản chiếu) nằm ngoài, đúng tinh thần của `eval.sh`.

Ô này chỉ in gợi ý. Muốn dùng thì chép giá trị vào `ENV_SCOPE` ở ô cấu hình, đổi `BASELINE_NAME` (ví dụ `rtsplatting_scope`), rồi chạy lại từ ô cấu hình. Báo cáo cả hai cấu hình sẽ cho reviewer thấy baseline không bị chỉnh kém.

In [ ]:
import json
import os
import sys

import numpy as np
from plyfile import PlyData

sys.path.insert(0, REPO_DIR)
from scene.colmap_loader import qvec2rotmat, read_extrinsics_binary

glass_ply = os.path.join(RUN_ROOT, 'full', 'glass_points.ply')
if not os.path.exists(glass_ply):
    print(f'Không có {glass_ply}: cần chạy VIGOR-GS full trước (notebook colab_vsgd_0758.ipynb).')
else:
    vertex = PlyData.read(glass_ply)['vertex']
    glass = np.stack([vertex['x'], vertex['y'], vertex['z']], axis=1)
    with open(os.path.join(SCENE_DIR, 'colmap', 'split.json')) as f:
        train_names = set(json.load(f)['train'])
    extrinsics = read_extrinsics_binary(os.path.join(SCENE_DIR, 'colmap', 'sparse', '0', 'images.bin'))
    centers = np.array([-qvec2rotmat(e.qvec).T @ e.tvec for e in extrinsics.values() if os.path.basename(e.name) in train_names])

    center = np.median(glass, axis=0)
    radius = float(np.median(np.linalg.norm(centers - center, axis=1)))
    inside = float(np.mean(np.linalg.norm(glass - center, axis=1) < radius))
    print(f'{len(glass)} điểm kính, {len(centers)} camera train')
    print(f'Gợi ý: ENV_SCOPE = [{center[0]:.3f}, {center[1]:.3f}, {center[2]:.3f}, {radius:.3f}]')
    print(f'  {inside * 100:.0f}% điểm kính nằm trong khối cầu này')

## Chạy baseline

Mỗi bước (`train`, `render`, `metrics`) ghi log vào `RUN_ROOT/<BASELINE_NAME>/log.txt` và để lại `.done_<bước>` khi xong. Khi Colab ngắt giữa chừng, chạy lại notebook từ đầu: các bước đã xong được bỏ qua. Bước `train` bị ngắt thì chạy lại từ đầu bước đó.

Lệnh train tương đương:

```
python train.py -s <scene> -m <RUN_ROOT>/rtsplatting -r 2 --eval --ignore_clean_images \
    --iterations 61000 --init_until_iter 700 --norm_loss_from_iter 700 --test_iterations 15250 30500 61000
```

Reader của repo nhận ra layout VIGOR (`colmap/frames.json`), nên baseline dùng đúng `split.json` và `colmap/masks` như VIGOR-GS. `--ignore_clean_images` chỉ để reader không nạp ảnh clean (baseline không dùng chúng, nạp vào chỉ tốn VRAM).

In [ ]:
import json
import os
import subprocess
import sys
import time

M = os.path.join(RUN_ROOT, BASELINE_NAME)
ENV = dict(os.environ, TQDM_MININTERVAL='60', PYTHONUNBUFFERED='1')  # tqdm in ra mỗi 60 s thay vì hàng nghìn dòng
if os.path.isdir(STUB_DIR):
    ENV['PYTHONPATH'] = STUB_DIR + os.pathsep + ENV.get('PYTHONPATH', '')
CONFIG = {
    'method': 'RT-Splatting (train.py)',
    'scene': 'VSGD/0758',
    'resolution': RESOLUTION,
    'iterations': ITERS,
    'init_until_iter': INIT_UNTIL_ITER,
    'norm_loss_from_iter': NORM_LOSS_FROM_ITER,
    'env_scope': ENV_SCOPE,
    'xyz_axis': XYZ_AXIS if ENV_SCOPE is not None else None,
    'data_device': DATA_DEVICE,
    'extra': EXTRA,
}


def run(args, log_path):
    """Run one repo script, stream its output here and into log_path; raise if it fails."""
    print('$ python', ' '.join(args), flush=True)
    start = time.time()
    with open(log_path, 'a') as log:
        log.write(f'\n$ python {" ".join(args)}\n')
        log.flush()
        proc = subprocess.Popen([sys.executable, *args], cwd=REPO_DIR, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line)
            print(line, end='', flush=True)
        code = proc.wait()
    print(f'  -> {(time.time() - start) / 60:.1f} phút', flush=True)
    if code:
        raise RuntimeError(f'{args[0]} lỗi (exit code {code}); xem {log_path}')


def step(name, args):
    marker = os.path.join(M, f'.done_{name}')
    if os.path.exists(marker):
        print(f'[{name}] đã xong trước đó, bỏ qua')
        return
    run(args, os.path.join(M, 'log.txt'))
    with open(marker, 'w') as f:
        f.write(time.strftime('%Y-%m-%d %H:%M:%S') + '\n' + CODE_VERSION)


def check_config():
    path = os.path.join(M, 'notebook_config.json')
    if os.path.exists(path):
        with open(path) as f:
            saved = json.load(f)
        if saved != json.loads(json.dumps(CONFIG)):
            raise RuntimeError(f'{M} đã được chạy với cấu hình khác:\n  cũ: {saved}\n  mới: {CONFIG}\nĐổi BASELINE_NAME, hoặc xoá thư mục này để chạy lại.')
    else:
        with open(path, 'w') as f:
            json.dump(CONFIG, f, indent=2)


os.makedirs(M, exist_ok=True)
check_config()
start = time.time()

train_args = ['train.py', '-s', SCENE_DIR, '-m', M, '-r', str(RESOLUTION), '--eval', '--ignore_clean_images', '--data_device', DATA_DEVICE]
train_args += ['--iterations', str(ITERS), '--init_until_iter', str(INIT_UNTIL_ITER), '--norm_loss_from_iter', str(NORM_LOSS_FROM_ITER)]
train_args += ['--test_iterations', *[str(i) for i in sorted({ITERS // 4, ITERS // 2, ITERS})]]
if ENV_SCOPE is not None:
    train_args += ['--env_scope_center', *map(str, ENV_SCOPE[:3]), '--env_scope_radius', str(ENV_SCOPE[3]), '--xyz_axis', *map(str, XYZ_AXIS)]
step('train', train_args + list(EXTRA))
step('render', ['render.py', '-m', M, '--skip_train', '--skip_mesh'])
step('metrics', ['metrics.py', '-m', M])
print(f'===== RT-Splatting: xong ({(time.time() - start) / 60:.1f} phút)', flush=True)

## So sánh với VIGOR-GS (split test)

Mọi dòng đều được `metrics.py` chấm trên PNG sRGB so với **ảnh gốc có phản chiếu**. Cột `Masked …` nhân cả hai ảnh với mask rồi tính trên toàn ảnh, nên cao hơn PSNR thật trong vùng kính (xem `exp_run/VSGD_0758/analysis_vi.md`, mục 2.1b).

Hai cột cuối so từng cặp view với RT-Splatting: `Δ PSNR` là trung bình chênh lệch theo cặp, `thắng` là số view mà biến thể có PSNR cao hơn RT-Splatting.

Cách đọc bảng:

- `full` so với RT-Splatting: VIGOR-GS có hơn baseline thật trên ảnh tổng hợp hay không.
- `no_separation` so với RT-Splatting: cấu hình "2DGS `p·o` + deformation, không có phần nào của VIGOR" khác baseline thật bao nhiêu. Phần chênh lệch này là công của deformation và các khác biệt cài đặt, không phải của VIGOR-GS.
- Hiệu `full − no_separation` (đã có trong phân tích trước) là phần mà cơ chế VIGOR-GS thêm vào hoặc làm mất đi.

In [ ]:
import json
import os

import numpy as np
import pandas as pd

VIGOR_VARIANTS = {
    'full': 'VIGOR-GS đầy đủ',
    'no_factorization': '(1) Không factorization p·o',
    'no_separation': '(2a) Một field trên ảnh gốc',
    'shared_field': '(2b) Không có G_R riêng',
    'no_virtual_camera': '(3) Không camera ảo',
    'no_fresnel': '(4) Không Fresnel',
}
ENTRIES = {BASELINE_NAME: 'RT-Splatting (baseline)', **VIGOR_VARIANTS}


def latest_method(results):
    return max(results, key=lambda name: int(name.rsplit('_', 1)[-1]))  # ours_<iter> / direct_<iter>: iteration lớn nhất


def load(key):
    folder = os.path.join(RUN_ROOT, key)
    if not os.path.exists(os.path.join(folder, 'results.json')):
        return None, None, None
    with open(os.path.join(folder, 'results.json')) as f:
        results = json.load(f)
    method = latest_method(results)
    with open(os.path.join(folder, 'per_view.json')) as f:
        per_view = json.load(f)[method]['PSNR']
    return method, results[method], per_view


_, _, baseline_views = load(BASELINE_NAME)
rows = []
for key, label in ENTRIES.items():
    method, scores, views = load(key)
    if method is None:
        rows.append({'phương pháp': label, 'thư mục': key, 'method': '(chưa chạy)'})
        continue
    row = {'phương pháp': label, 'thư mục': key, 'method': method, **scores}
    if baseline_views is not None and key != BASELINE_NAME:
        common = sorted(set(views) & set(baseline_views))
        deltas = np.array([views[name] - baseline_views[name] for name in common])
        row['Δ PSNR so với RT-Splatting'] = deltas.mean()
        row['thắng / số view'] = f'{int((deltas > 0).sum())}/{len(common)}'
    rows.append(row)

table = pd.DataFrame(rows)
table.to_csv(os.path.join(RUN_ROOT, 'summary_with_rtsplatting.csv'), index=False)
display(table.round(4))

## Xem một frame test

Hàng trên: ảnh gốc, ảnh render của RT-Splatting và của VIGOR-GS `full`. Hàng dưới: hai cách tách lớp, tức lớp truyền qua (`transmitted`) và lớp phản xạ (`specular`) của RT-Splatting, so với `B` (direct) và `α·R` của VIGOR-GS. Đây chỉ là so sánh định tính: các lớp của hai phương pháp không được định nghĩa giống hệt nhau.

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
from PIL import Image

FRAME = '00026.png'  # chỉ số frame trong split test (00026 là view dùng trong analysis_vi.md)


def method_dir(key):
    found = sorted(glob.glob(os.path.join(RUN_ROOT, key, 'test', '*', 'renders')), key=lambda p: int(os.path.basename(os.path.dirname(p)).rsplit('_', 1)[-1]))
    return os.path.dirname(found[-1]) if found else None


base, full = method_dir(BASELINE_NAME), method_dir('full')
panels = []
if base:
    panels += [('Ảnh gốc (gt)', os.path.join(base, 'gt', FRAME)), ('RT-Splatting: render', os.path.join(base, 'renders', FRAME))]
if full:
    panels += [('VIGOR-GS full: render', os.path.join(full, 'renders', FRAME))]
if base:
    panels += [('RT-Splatting: transmitted', os.path.join(base, 'vis', 'transmitted_' + FRAME)), ('RT-Splatting: specular', os.path.join(base, 'vis', 'specular_' + FRAME))]
if full:
    panels += [('VIGOR-GS full: B (direct)', os.path.join(full, 'vis', 'direct_' + FRAME)), ('VIGOR-GS full: α·R', os.path.join(full, 'vis', 'reflection_contribution_' + FRAME))]
panels = [(t, p) for t, p in panels if os.path.exists(p)]
if panels:
    cols = 4
    rows_ = (len(panels) + cols - 1) // cols
    fig, axes = plt.subplots(rows_, cols, figsize=(5 * cols, 3.2 * rows_), squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')
    for ax, (title, path) in zip(axes.ravel(), panels):
        ax.imshow(Image.open(path))
        ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print('Chưa có ảnh render nào.')